# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



In [6]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.


In [7]:
# Business/Research question: 
    # Is there an association between the movies that average lower ratings and the keywords associated with those movies?
# Why this question is important: 
   #Understanding the association between lower-rated movies and their keywords can provide information into what types of content may not appeal to audiences. 
   #This information can be valuable for filmmakers to improve future movie green light decisions or set expectations for marketing campaigns.


In [8]:
#Table/files used:
# (FILENAME): (DESCRIPTION) - (Reasoning for using this table/file)
# 1. movies_metadata.csv: title, genre, release year, language - This is the base movie table/file that contains the movieId and tmdbId, which are used to join the other tables/files together.
# 2. ratings.csv: individual user ratings- Critical to mr question since it is used to compute each movie's average rating and rating count
# 3. keywords_curated.csv: keyword list for each movie- Another critical table/file since it is used to identify the keywords associated with each movie, which is the other half of my research question.
# 4. links.csv: ID crosswalk - I may need this table/file to join the other tables/files together, but it is not central to my research question.
# 5. credits_curated.csv: director and top cast member information - not central to the question, but keeping as a reference in case i want to expand my research question to include director or cast member information in the future.


# Count of ratings for each movie
ratings_per_movie = ratings.groupby("movieId").size()
ratings_per_movie.describe()  


count    9025.000000
mean       11.059280
std        24.095433
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
dtype: float64

In [9]:
# How many movies have only 1 rating vs. 10+ ratings
print("Movies with 1 rating:", (ratings_per_movie == 1).sum())
print("Movies with 10+ ratings:", (ratings_per_movie >= 10).sum())

Movies with 1 rating: 3042
Movies with 10+ ratings: 2241


In [10]:
#Movie keywords missingn.
keywords["keywords_list"].isna().sum()

np.int64(761)

In [11]:
#Ratings value counts. Note this includes ratings for all movies to include many for the same movie by different users. 
# I will need to filter this down to only include movies with 10+ ratings to ensure that the average rating is more reliable.
ratings["rating"].value_counts().sort_index()

rating
0.5     1097
1.0     3322
1.5     1685
2.0     7263
2.5     4446
3.0    20039
3.5    10512
4.0    28705
4.5     7698
5.0    15043
Name: count, dtype: int64

In [12]:
#Row meaning of each important table:
#movies_metadata.csv: Each row represents a movie, with columns for title, genre, release year, language, and unique identifiers (movieId and tmdbId).
#ratings.csv: Each row represents a user rating for a specific movie, with columns for userId, movieId, rating value, and timestamp of the rating.
#keywords_curated.csv: Each row represents one movie's keyword list. Very important when I am trying to associated the most common keywords to the worst rated movies.
#links.csv: Each row represents a mapping between movieId and tmdbId, allowing for cross-referencing between different datasets.



In [13]:
#Early data quality issues:
# 1. Number of ratings per movie. The median movie has only 3 ratings and 3,042 movies have only 1 rating. 2241 movies have 10 or more ratings. 
#  For now I think my analysis will focus on movies with a minimum of 10 ratings to ensure that the average rating is more reliable.
#2. 761 movies are missing keywords. I will need to decide how to handle these movies in my analysis.
#3. In movies_metadata.csv, the release_date is stored as a string and not a date. I will need to convert this column to a date format.
#4. There are some movies with duplicate titles but different movieId's. I will need to decide how to handle these movies in my analysis.
#5. I have concerns about the keyword format. The file uses pipe characters to separate keywords. I will need to decide how to handle these cases in my analysis.
#6. I will need to consider if and how I want to group the keywords into broader categories to make the analysis more manageable and meaningful. This is a big one for me to think about.

In [14]:
#initial cleaning plan:
# Step 1 - Protect the raw data by creating a copy of the original csv files to work with. This will ensure that the original data remains unchanged and can be referenced if needed.
# Step 2 - Structural issues:
#   Issue: Release date is stored as a string.
#   Plan: Convert release_date to a datetime format.
#   Issue: Keywords are stored as a pipe-separated string.
#   Plan: Split the keywords into a list of individual keywords for each movie.
# Step 3: Handle duplicates:
#   Issue: Some movies have duplicate titles but different movieId's.
#   Plan: Decide whether to keep all duplicates or different movies with the same title then take the appropriate action.
# Step 4: Standardize keywords:
#   Issue: Keywords may have inconsistent formatting and many may be unique or rare, making it difficult to analyze.
#   Plan: Decide I want a minimum frequency threshold for keywords to be included in the analysis. Maybe 10 appearances in the dataset.
#    Consider grouping similar keywords into broader categories to make the analysis more manageable and meaningful. I will need to document the rules for grouping keywords and ensure that the grouping is consistent across the dataset.

In [15]:
#Possible target/outcome:
# 1. Identify movies with at least 10 user ratings in a new column or table to filter the dataset to only include movies with a minimum of 10 ratings. 
# 2. Compute the average rating for each movie and store it in a new column or table.
# 3. Movies with less than 10 ratings will get a NaN value.
# 4. I need to identify a way to identify which of these qualifying averages ratings fall below the requirement of under 3.0.
# 5. I will need to identify the most common keywords associated with these lower-rated movies and determine if there is a pattern or association between the keywords and the lower ratings.

In [16]:
#Ethics, privacy, or governance concerns:
# 1. Real names are included in the data including the names of directors and top cast memebers. I will need to consider if this is sensitve data and how I will handle it in my analysis.
# 2. I will keep the raw files unchanged and only clean working copies.
# 3. Proper documentation of data cleaning or analysis will be important along the way.
# 4. I believe I will need to make some judgement calls about grouping the review ratings and especially when grouping the keywords into broader categories. I will need to document my decisions and ensure that they are consistent across the dataset.
# 5. When creating the broader keyword categories I will need to ensure I am not discriminating against a specific group of people. Examples of this could be based on gender, race, or religious related keywords.


In [17]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [18]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



### SQL/DB Check using Neon

In [19]:
#Connect to the database and test the connection
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env", override=True)
db_url = os.getenv("PROJECT_DATABASE_URL")
print("URL found:", db_url is not None)

engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

URL found: True
PostgreSQL 18.6 (ac70818) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [20]:
#Upload the data from the CSV file into a pandas DataFrame
import pandas as pd

ratings_raw = pd.read_csv("../data/ratings.csv")
links_raw = pd.read_csv("../data/links.csv")
print("ratings rows:", len(ratings_raw), "| links rows:", len(links_raw))

ratings_raw.to_sql("ratings", engine, if_exists="replace", index=False,
                   chunksize=5000, method="multi")
links_raw.to_sql("links", engine, if_exists="replace", index=False,
                 chunksize=5000, method="multi")

with engine.connect() as conn:
    for t in ["ratings", "links"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"{t} rows in Neon:", n)

ratings rows: 99810 | links rows: 9125
ratings rows in Neon: 99810
links rows in Neon: 9125


In [ ]:
# Upload the remaining project tables to Neon so they can be queried with SQL.

other_tables = {
    "movies_metadata": pd.read_csv("../data/movies_metadata.csv"),
    "credits": pd.read_csv("../data/credits_curated.csv"),
    "keywords": pd.read_csv("../data/keywords_curated.csv"),
}

for name, df in other_tables.items():
    df.to_sql(name, engine, if_exists="replace", index=False, chunksize=5000, method="multi")
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {name}")).scalar()
    print(f"{name}: {len(df)} rows in Pandas | {n} rows in Neon")

movies_metadata: 9082 rows in Pandas | 9082 rows in Neon
credits: 9082 rows in Pandas | 9082 rows in Neon
keywords: 9082 rows in Pandas | 9082 rows in Neon


In [ ]:
# Helper function for running SQL against the Neon database.
# If a database connection is available (my .env), it runs the query and saves the # result as a CSV in outputs/sql_results/.

from pathlib import Path

SQL_RESULTS_DIR = Path("../outputs/sql_results")
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def run_project_sql(query, name):
    """Run SQL on Neon if a connection is available; otherwise load the saved result."""
    out_path = SQL_RESULTS_DIR / f"{name}.csv"
    url = os.getenv("PROJECT_DATABASE_URL")
    if url:
        result = pd.read_sql(text(query), create_engine(url))
        result.to_csv(out_path, index=False)
        print(f"Ran query on Neon and saved -> {out_path}")
    else:
        result = pd.read_csv(out_path)
        print(f"No database connection; loaded saved result <- {out_path}")
    return result

In [ ]:
# SQL aggregation: roll the rating-event-level ratings table up to one row per movie.
# For each movieId, count the ratings and calculate the average rating, keeping only movies with at least 10 ratings (the threshold used in my research question).

sql_rating_summary = run_project_sql("""
    SELECT "movieId",
           COUNT(*) AS rating_count,
           ROUND(AVG(rating)::numeric, 4) AS avg_rating
    FROM ratings
    GROUP BY "movieId"
    HAVING COUNT(*) >= 10
    ORDER BY "movieId";
""", name="rating_summary_min10")

print("Movies with 10+ ratings (SQL):", len(sql_rating_summary))
sql_rating_summary.head()


Ran query on Neon and saved -> ..\outputs\sql_results\rating_summary_min10.csv
Movies with 10+ ratings (SQL): 2241


,movieId,rating_count,avg_rating
0,1,247,3.8725
1,2,107,3.4019
2,3,59,3.1610
3,4,13,2.3846
4,5,56,3.2679


In [ ]:
# Validation: rebuild the same movie-level rating summary in Pandas and compare it to the SQL result. Matching movie counts, rating counts, and average ratings confirm the
# ratings were aggregated correctly before they get merged into the movie-level ABT.

pd_rating_summary = (
    ratings_raw.groupby("movieId")["rating"]
    .agg(rating_count="size", avg_rating="mean")
    .reset_index()
    .query("rating_count >= 10")
)

check = pd_rating_summary.merge(sql_rating_summary, on="movieId", suffixes=("_pd", "_sql"))
print("Movies (Pandas):", len(pd_rating_summary), "| Movies (SQL):", len(sql_rating_summary))
print("Counts match for every movie:", (check["rating_count_pd"] == check["rating_count_sql"]).all())
print("Max avg difference:", (check["avg_rating_pd"] - check["avg_rating_sql"]).abs().max())

Movies (Pandas): 2241 | Movies (SQL): 2241
Counts match for every movie: True
Max avg difference: 5.000000000032756e-05


**SQL validation result:** The SQL query on the Neon database aggregated 99,810 rating events to one row per movie and found 2,241 movies with at least 10 ratings. The same aggregation in Pandas produced an identical set of movies with matching rating counts, and average ratings. This confirms the ratings table is correctly rolled up to the movie level before it is merged into the movie level ABT. The merge will not duplicate movie rows.

In [33]:
# Step 1: Cleaning: work on copies so the raw data remains unchanged and can be referenced if needed.
movies_clean = movies.copy()
keywords_clean = keywords.copy()
credits_clean = credits.copy()

movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")
print("Release dates that could not be converted to datetime:", movies_clean["release_date"].isna().sum())

#Standardize keywords: Split the keywords into a list of individual keywords for each movie.
keywords_clean["keywords_list"] = (keywords_clean["keywords_list"].str.split("|")
.apply(lambda kws: "|".join(k.strip() for k in kws) if isinstance(kws, list) else np.nan)
)
keywords_clean["keywords_count"] = keywords_clean["keywords_list"].str.split("|").str.len().fillna(0).astype(int)
keywords_clean["has_keywords"] = keywords_clean["keywords_list"].notna().astype(int)

print("Rows sharing a title:", movies_clean["title"].duplicated(keep=False).sum())
print("Rows sharing title and release year:", movies_clean[["title", "release_year"]].duplicated(keep=False).sum())


Release dates that could not be converted to datetime: 0
Rows sharing a title: 521
Rows sharing title and release year: 2


In [34]:
# Step 2: Aggregate ratings to one row per movie.
rating_summary = (
    ratings.groupby("movieId")
    .agg(user_rating_mean=("rating", "mean"), user_rating_count=("rating", "size"), user_rating_std=("rating", "std"), distinct_user_count=("userId", "nunique"))
    .reset_index()
)
print("rating_summary rows:", len(rating_summary), "| duplicate movieIds:", rating_summary["movieId"].duplicated().sum())

rating_summary rows: 9025 | duplicate movieIds: 0


In [35]:
#step 3: merge tables. movies is the base table, and I will merge the other tables into it to create the final ABT for analysis.
def merge_check(df, label):
    print(f"{label}: {len(df)} rows | duplicate movieIds: {df['movieId'].duplicated().sum()}")
    
print(f"Start (movies): {len(movies_clean):,} rows")
abt = movies_clean.merge(rating_summary, on="movieId", how="left", validate="1:1")
merge_check(abt, "+ rating_summary on movieId")
abt = abt.merge(keywords_clean, on="tmdbId", how="left", validate="1:1")
merge_check(abt, "+ keywords on tmdbId")
abt = abt.merge(credits_clean, on="tmdbId", how="left", validate="1:1")
merge_check(abt, "+ credits on tmdbId")


Start (movies): 9,082 rows
+ rating_summary on movieId: 9082 rows | duplicate movieIds: 0
+ keywords on tmdbId: 9082 rows | duplicate movieIds: 0
+ credits on tmdbId: 9082 rows | duplicate movieIds: 0


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [27]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [28]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,,,,
1,tmdbId,int64,,,,
2,imdb_id,str,,,,
3,title,str,,,,
4,original_title,str,,,,


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [29]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [30]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [31]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [32]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
